## Kaggriculture experiment runner and analysis

This notebook is the front end for the evaluation pipeline in this project.

The main analytical source is `episodes.parquet`: one row per player per game. Means and other summaries are derived from those episode rows rather than replacing them.

Run this notebook from the project directory containing `main.py`, `dynamic_all_agent_v1.py`, and `evaluation_pipeline/`.


### 1. Setup

The evaluation package requires `pandas` and `pyarrow`; running new games also requires `kaggle-environments`. If Parquet support is missing, install it once with `pip install pyarrow` in your project environment.


In [ ]:
from pathlib import Path
import sys
import json

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Allow the notebook to work from the project root or from a notebooks/ subfolder.
PROJECT_ROOT = Path.cwd().resolve()
if (not (PROJECT_ROOT / "evaluation_pipeline" / "metrics.py").exists()
        and (PROJECT_ROOT.parent / "evaluation_pipeline" / "metrics.py").exists()):
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "evaluation_pipeline" / "metrics.py").exists():
    raise RuntimeError("Could not find metrics.py. Start the notebook from the Kaggriculture project root "
                       "or from a direct notebooks/ child folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

from evaluation_pipeline import parse_run
from evaluation_pipeline.metrics import evaluate_and_log

print(f"Project root: {PROJECT_ROOT}")


In [ ]:
import importlib

import runner


def _load_agent(name):
    policy = importlib.import_module(f"policies.{name}")
    policy = importlib.reload(policy)
    return policy.agent


def reload_agents(experiment):
    global TEST_AGENT, BASELINE_AGENT
    global TEST_AGENT_NAME, BASELINE_AGENT_NAME

    importlib.reload(runner)

    TEST_AGENT_NAME, BASELINE_AGENT_NAME = experiment

    TEST_AGENT = _load_agent(TEST_AGENT_NAME)
    BASELINE_AGENT = _load_agent(BASELINE_AGENT_NAME)

    print(f"Agents reloaded: {TEST_AGENT_NAME} vs {BASELINE_AGENT_NAME}")

In [ ]:
# Metrics constants
HEADLINE_METRICS = ["final_cash",
                    "margin",
                    "win",
                    "watering_deaths",
                    "animal_escapes",
                    "estimated_shed_overflow_units",
                    "productive_utilization",
                    "movement_rate",
                    "pass_rate",
                    "known_noop_rate",
                    "travel_per_productive_action",
                    "market_orders_over_limit"]

RESOURCE_METRICS = ["plants_planted",
                    "crop_units_harvested",
                    "watering_deaths",
                    "natural_decays",
                    "crop_units_lost_to_decay",
                    "missed_water_days",
                    "water_adherence_rate",
                    "critical_water_events",
                    "critical_water_rescues",
                    "critical_water_failures",
                    "animals_bought_observed",
                    "animals_placed",
                    "animal_escapes",
                    "unfed_animal_days",
                    "unfed_production_days",
                    "care_bonus_units_forfeited",
                    "feed_adherence_rate",
                    "critical_feed_events",
                    "critical_feed_rescues",
                    "critical_feed_failures",
                    "animal_product_units_harvested",
                    "fertilizer_collected",
                    "fertilizer_collection_opportunities_missed",
                    "estimated_shed_overflow_units",
                    "stranded_animals_end",
                    "stranded_animal_cost_end",
                    "unused_seeds_end",
                    "unused_seed_cost_end",
                    "unsold_sellable_units_end",
                    "terminal_sellable_value_at_current_prices"]

PRODUCTION_METRICS = ["mean_land_occupancy",
                      "crop_tile_days",
                      "animal_tile_days",
                      "weed_tile_days",
                      "empty_structure_tile_days",
                      "crop_units_harvested",
                      "animal_product_units_harvested",
                      "fertilizer_collected",
                      "crop_units_per_tile_day",
                      "animal_product_units_per_tile_day",
                      "harvested_units_per_worker_turn"]

LABOR_METRICS = ["worker_turns",
                 "hands_hired_observed",
                 "hire_cost_observed",
                 "productive_actions_effective",
                 "movement_actions",
                 "logistics_actions",
                 "pass_actions",
                 "known_noop_actions",
                 "productive_utilization",
                 "movement_rate",
                 "pass_rate",
                 "known_noop_rate",
                 "travel_per_productive_action",
                 "harvested_units_per_worker_turn",
                 "productive_actions_per_hire"]

MARKET_METRICS = ["market_orders_submitted",
                  "market_orders_over_limit",
                  "market_turns_over_limit",
                  "animals_bought_observed",
                  "max_shed_inventory",
                  "mean_shed_inventory",
                  "estimated_shed_overflow_units",
                  "unsold_sellable_units_end",
                  "terminal_sellable_value_at_current_prices"]

LOSS_EVENTS = ["PLANT_DIED_UNWATERED",
               "PLANT_DIED_SAME_DAY_UNWATERED",
               "ANIMAL_ESCAPED",
               "SHED_OVERFLOW_ESTIMATED",
               "MARKET_ORDER_DROPPED_LIMIT",
               "CROP_UNITS_LOST_TO_DECAY",
               "UNFED_ANIMAL_PRODUCTION_DAY",
               "FERTILIZER_COLLECTION_OPPORTUNITY_MISSED"]

PRODUCTION_EVENTS = ["CROP_HARVESTED",
                     "ANIMAL_PRODUCT_HARVESTED",
                     "FERTILIZER_COLLECTED"]

DROP_COLUMNS = ["run_id", "episode_id", "source_file", "role", "opponent_role", "resolved_seed"]


### 2. Run a new experiment

This produces a new directory under `evaluations/` containing compressed raw episodes and processed Parquet tables. Each seed is played twice so the test agent appears once in each player position.


In [ ]:
# Test vs baseline agent names. These are the two agents compared in the evaluation.
EXPERIMENT = ("gbt_labor", "heuristic_v2")

reload_agents(experiment=EXPERIMENT)

SEEDS = 10
EPISODE_STEPS = 720
OUTPUT_ROOT = PROJECT_ROOT / "evaluations"
MAX_WORKERS = 8
RUN_NAME = None

run_dir, frames = evaluate_and_log(TEST_AGENT,
                                   BASELINE_AGENT,
                                   SEEDS,
                                   TEST_AGENT_NAME,
                                   BASELINE_AGENT_NAME,
                                   output_root=OUTPUT_ROOT,
                                   run_name=RUN_NAME,
                                   max_workers=MAX_WORKERS)

run_dir = Path(run_dir)
print(f"\nRun directory: {run_dir}")

# Alternative: load an existing experiment instead of running a new one.
# EXISTING_RUN_DIR = PROJECT_ROOT / "evaluations" / "heuristic_v3_vs_heuristic_v2_..."
# run_dir = Path(EXISTING_RUN_DIR)
# frames = parse_run(run_dir, write_outputs=False)
# print(f"Loaded: {run_dir}")

#### 3. Inspect what was captured

The five detailed tables serve different purposes:

- **episodes** — one row per player per game; primary benchmarking table.
- **events** — sparse state transitions such as plant death, animal escape, harvests, decay losses, and overflow estimates.
- **worker_actions** — one row per submitted farmer/hand action.
- **market_orders** — one row per submitted market order, including queue-limit position and market context.
- **daily_states** — one compact strategic snapshot per player per day.

`summary` contains derived aggregate statistics and paired deltas.


In [ ]:
episodes = frames["episodes"]
events = frames["events"]
worker_actions = frames["worker_actions"]
market_orders = frames["market_orders"]
daily_states = frames.get("daily_states", pd.DataFrame())
summary = frames["summary"]

tables = {"episodes": episodes,
          "events": events,
          "worker_actions": worker_actions,
          "market_orders": market_orders,
          "daily_states": daily_states,
          "summary": summary}

sizes = pd.DataFrame([{"table": name, "rows": len(df), "columns": len(df.columns)}
                      for name, df in tables.items()])

# display(sizes)


## Headline metrics comparison

This is the compact test-vs-baseline view. Paired columns compare the agents within the exact same match rather than only comparing independent overall means. `direction` identifies whether higher, lower, or neither direction is intrinsically preferable for a metric.


In [ ]:
headline = summary[summary["metric"].isin(HEADLINE_METRICS)].copy()
headline = headline.set_index("metric").reindex(HEADLINE_METRICS).reset_index()
headline_columns = ["metric",
                    "direction",
                    "test_mean",
                    "baseline_mean",
                    "delta",
                    "delta_pct",
                    "paired_delta_mean",
                    "paired_95ci_low",
                    "paired_95ci_high",
                    "paired_improved_rate"]

headline_columns = [column for column in headline_columns if column in headline.columns]
display(headline[headline_columns].round(2))

### 5. Outcome distributions and head-to-head stability

Means can hide unstable agents. These views preserve the episode distribution, show the test agent's margin in every match, measure player-position effects, and collapse the two swapped-position games into one result per requested seed.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for agent_name, group in episodes.groupby("agent"):
    ax.hist(group["final_cash"].dropna(), bins=12, alpha=0.5, label=agent_name)

ax.set_title("Final cash distribution")
ax.set_xlabel("Final cash")
ax.set_ylabel("Episodes")
ax.legend()
plt.show()


In [ ]:
test_episodes = (episodes[episodes["role"] == "test"]
                 .sort_values(["requested_seed", "player"])
                 .copy())

fig, ax = plt.subplots(figsize=(10, 5))
ax.axhline(0, linewidth=1)
ax.plot(range(len(test_episodes)), test_episodes["margin"].to_numpy(), marker="o", linewidth=1)
ax.set_title("Test-agent margin in every paired match")
ax.set_xlabel("Match")
ax.set_ylabel("Final cash margin vs opponent")
ax.legend([f"{TEST_AGENT_NAME} vs {BASELINE_AGENT_NAME}"])
plt.show()

cols = ["match_id", "requested_seed", "player", "agent", "opponent",
        "final_cash", "opponent_cash", "margin", "win"]
display(test_episodes[cols].reset_index(drop=True))


In [ ]:
position_view = (test_episodes.groupby("player")[["final_cash", "margin", "win"]]
                 .agg(["mean", "median", "std"]))

display(Markdown("**Results by player position**"))
display(position_view)

seed_pairs = (test_episodes.pivot_table(index="requested_seed",
                                        columns="player",
                                        values="margin",
                                        aggfunc="first")
              .rename(columns={0: "player_0_margin", 1: "player_1_margin"})
              .reset_index())

for column in ["player_0_margin", "player_1_margin"]:
    if column not in seed_pairs.columns:
        seed_pairs[column] = pd.NA

seed_pairs["mean_margin"] = seed_pairs[["player_0_margin", "player_1_margin"]].mean(axis=1)

display(Markdown("**Paired result by requested seed**"))
display(seed_pairs)


## Detailed Metrics

### 6. Resource loss and care quality

These metrics separate catastrophic failures from opportunity losses. Natural crop decay remains distinct from harvestable units actually lost to decay, and skipped animal feeding is separated from skipped production days and forfeited care bonus.


In [ ]:
available = [metric for metric in RESOURCE_METRICS if metric in episodes.columns]
resource_view = episodes.groupby("agent")[available].mean().T
resource_view.index.name = "metric"
display(resource_view)

In [ ]:
if "event" in events.columns:
    avoidable_losses = events[events["event"].isin(LOSS_EVENTS)].copy()

    with pd.option_context("display.max_rows", None):
        display(avoidable_losses.drop(columns=DROP_COLUMNS, errors="ignore")
                                .sort_values(["agent", "match_id", "step"])
                                .reset_index(drop=True))
else:
    print("No event rows were produced for this run.")


### 7. Worker efficiency

`productive_utilization` uses actions that the parser could verify as effective. The additional views below show whether labor efficiency changes by worker index and by phase of the season, which is useful for testing the marginal value of later hires.


In [ ]:
available = [metric for metric in LABOR_METRICS if metric in episodes.columns]
labor_view = episodes.groupby("agent")[available].mean().T
labor_view.index.name = "metric"
display(labor_view)

In [ ]:
if not worker_actions.empty and "category" in worker_actions.columns:
    action_mix = (worker_actions.groupby(["agent", "category"])
                  .size()
                  .unstack(fill_value=0))

    action_mix_pct = action_mix.div(action_mix.sum(axis=1), axis=0)

    # display(Markdown("**Worker action counts**"))
    # display(action_mix)

    display(Markdown("**Worker action shares**"))
    display(action_mix_pct)


In [ ]:
test_worker = worker_actions[worker_actions["role"] == "test"].copy()

if not test_worker.empty:
    test_worker["effective_productive"] = (
        ((test_worker["category"] == "productive") & (test_worker["success"] == True))
        .astype(int)
    )

    worker_view = (test_worker.groupby(["worker_index", "worker_type"])
                   .agg(worker_turns=("action", "size"),
                        productive_actions=("effective_productive", "sum"),
                        productive_rate=("effective_productive", "mean"),
                        movement_rate=("category", lambda x: (x == "movement").mean()),
                        pass_rate=("category", lambda x: (x == "idle").mean()))
                   .reset_index())

    display(Markdown("**Labor by worker index**"))
    display(worker_view)


In [ ]:
if not test_worker.empty and "decision_day" in test_worker.columns:
    test_worker["phase"] = pd.cut(test_worker["decision_day"],
                                  bins=[-1, 9, 19, 29],
                                  labels=["early: days 0-9",
                                          "middle: days 10-19",
                                          "late: days 20-29"])

    worker_phase = pd.crosstab(test_worker["phase"],
                               test_worker["category"],
                               normalize="index")

    display(Markdown("**Worker action share by phase of season**"))
    display(worker_phase)


### 8. Land, production, and strategy evolution

These views normalize output against land/time, break production down by item, and show how the farm's strategic state evolves over the season.


In [ ]:
available = [metric for metric in PRODUCTION_METRICS if metric in episodes.columns]
production_view = episodes.groupby("agent")[available].mean().T
production_view.index.name = "metric"
display(production_view)

In [ ]:
if not events.empty and {"event", "item", "amount"}.issubset(events.columns):
    production_mix = (
        events[(events["role"] == "test") & events["event"].isin(PRODUCTION_EVENTS)]
        .groupby(["event", "item"], dropna=False)["amount"]
        .sum()
        .sort_values(ascending=False)
        .rename("units")
        .reset_index()
    )

    display(Markdown("**Test-agent production mix**"))
    display(production_mix)

In [ ]:
if not daily_states.empty:
    test_daily = daily_states[daily_states["role"] == "test"].copy()

    daily_strategy = (test_daily.groupby("day")
                      .agg(mean_cash=("cash", "mean"),
                           mean_land_occupancy=("land_occupancy", "mean"),
                           mean_crop_tiles=("crop_tiles", "mean"),
                           mean_animal_tiles=("animal_tiles", "mean"),
                           mean_shed_units=("shed_units", "mean"),
                           mean_wheat_plants=("wheat_plants", "mean"),
                           mean_carrot_plants=("carrot_plants", "mean"),
                           mean_tomato_plants=("tomato_plants", "mean"),
                           mean_strawberry_plants=("strawberry_plants", "mean"),
                           mean_melon_plants=("melon_plants", "mean"),
                           mean_geese=("goose_count", "mean"),
                           mean_cows=("cow_count", "mean"),
                           mean_sheep=("sheep_count", "mean"))
                      .reset_index())

    display(Markdown("**Mean strategic state by day**"))
    display(daily_strategy)
else:
    print("No daily_states table was produced. Reparse the run after adding daily-state support to episode.py and parse.py.")

### 9. Market behavior

`within_order_limit` is exact: orders after the configured queue limit are known to be dropped. The grouped view shows which decisions are being truncated and, when parser support is present, the market price and inventory the policy saw when it submitted them.


In [ ]:
available = [metric for metric in MARKET_METRICS if metric in episodes.columns]
market_view = episodes.groupby("agent")[available].mean().T
market_view.index.name = "metric"
display(market_view)

In [ ]:
if not market_orders.empty and "within_order_limit" in market_orders.columns:
    dropped_orders = market_orders[
        (market_orders["role"] == "test")
        & (~market_orders["within_order_limit"].astype(bool))
    ].copy()

    print(f"Test-agent orders dropped by queue limit: {len(dropped_orders):,}")

    if not dropped_orders.empty:
        agg_spec = {"orders_dropped": ("order_index", "size")}

        if "market_price" in dropped_orders.columns:
            agg_spec["mean_market_price"] = ("market_price", "mean")

        if "market_inventory" in dropped_orders.columns:
            agg_spec["mean_market_inventory"] = ("market_inventory", "mean")

        dropped_order_view = (dropped_orders.groupby(["action", "item"], dropna=False)
                              .agg(**agg_spec)
                              .sort_values("orders_dropped", ascending=False)
                              .reset_index())

        display(Markdown("**Dropped market orders by decision type**"))
        display(dropped_order_view)

        with pd.option_context("display.max_rows", None):
            display(dropped_orders.drop(columns=DROP_COLUMNS, errors="ignore")
                                  .sort_values(["match_id", "step", "order_index"])
                                  .reset_index(drop=True))

### 10. Drill into one match

Choose any `match_id` from the episode table. This gives you the two episode rows plus the underlying events, worker actions, market orders, and daily strategic states for that exact game.


In [ ]:
MATCH_ID = test_episodes.iloc[0]["match_id"] if not test_episodes.empty else None
MATCH_ID

In [ ]:
if MATCH_ID is not None:
    display(Markdown(f"### Episode metrics — `{MATCH_ID}`"))
    display(episodes[episodes["match_id"] == MATCH_ID].T)

    display(Markdown("### Events"))
    display(events[events["match_id"] == MATCH_ID]
            .drop(columns=DROP_COLUMNS, errors="ignore")
            .reset_index(drop=True))

    display(Markdown("### Worker actions"))
    display(worker_actions[worker_actions["match_id"] == MATCH_ID]
            .drop(columns=DROP_COLUMNS, errors="ignore")
            .reset_index(drop=True))

    display(Markdown("### Market orders"))
    display(market_orders[market_orders["match_id"] == MATCH_ID]
            .drop(columns=DROP_COLUMNS, errors="ignore")
            .reset_index(drop=True))

    if not daily_states.empty:
        display(Markdown("### Daily strategic states"))
        display(daily_states[daily_states["match_id"] == MATCH_ID]
                .drop(columns=DROP_COLUMNS, errors="ignore")
                .reset_index(drop=True))


### 11. Find the worst episodes for a metric

Change `METRIC` to any column in `episodes`. This is useful for quickly finding failure cases to inspect in the detailed tables or raw JSON.


In [ ]:
METRIC = "animal_escapes"
N = 10

if METRIC not in episodes.columns:
    raise KeyError(f"Unknown metric: {METRIC}")

worst = (episodes[episodes["role"] == "test"]
         .sort_values(METRIC, ascending=False)
         .head(N))

display(worst[["match_id",
               "requested_seed",
               "player",
               "agent",
               "final_cash",
               "margin",
               METRIC]])


### 12. Compare saved experiments over time

This scans every processed evaluation under `evaluations/` and builds a lightweight run history from each `summary.csv`. It does not modify any experiment.


In [ ]:
def load_run_history(output_root=OUTPUT_ROOT):
    rows = []
    output_root = Path(output_root)

    if not output_root.exists():
        return pd.DataFrame()

    for summary_path in sorted(output_root.glob("*/processed/summary.csv")):
        run_path = summary_path.parent.parent
        metadata_path = run_path / "metadata.json"
        metadata = json.loads(metadata_path.read_text()) if metadata_path.exists() else {}
        saved_summary = pd.read_csv(summary_path)

        wanted = {"final_cash",
                  "margin",
                  "win",
                  "watering_deaths",
                  "animal_escapes",
                  "productive_utilization",
                  "travel_per_productive_action",
                  "market_orders_over_limit"}

        record = {"run": run_path.name,
                  "created_at_utc": metadata.get("created_at_utc"),
                  "test_agent": metadata.get("test_agent"),
                  "baseline_agent": metadata.get("baseline_agent"),
                  "seed_count": metadata.get("requested_seed_count"),
                  "match_count": metadata.get("match_count")}

        for _, row in saved_summary[saved_summary["metric"].isin(wanted)].iterrows():
            record[f"{row['metric']}_mean"] = row["test_mean"]
            record[f"{row['metric']}_delta"] = row["paired_delta_mean"]

        rows.append(record)

    return pd.DataFrame(rows)


history = load_run_history()
display(history.sort_values("created_at_utc", ascending=False) if not history.empty else history)


### 13. Raw files and reproducibility

Every game remains available under the run's `raw/` directory as gzip-compressed `env.toJSON()` output. The Parquet tables are derived artifacts and can be regenerated later if you add or change metrics.


In [ ]:
metadata_path = run_dir / "metadata.json"
metadata = json.loads(metadata_path.read_text())

display(pd.Series({"run_dir": str(run_dir),
                   "test_agent": metadata.get("test_agent"),
                   "baseline_agent": metadata.get("baseline_agent"),
                   "requested_seed_count": metadata.get("requested_seed_count"),
                   "match_count": metadata.get("match_count"),
                   "episode_steps": metadata.get("episode_steps"),
                   "seed_configuration_key": metadata.get("seed_configuration_key"),
                   "raw_format": metadata.get("raw_format")}))

raw_files = sorted((run_dir / "raw").glob("*.json.gz"))
print(f"Raw episodes retained: {len(raw_files)}")

if raw_files:
    print(f"Example: {raw_files[0]}")


## Render

In [ ]:
from kaggle_environments import make

env = make(
    "kaggriculture",
    configuration={
        "episodeSteps": EPISODE_STEPS,
        "seed": 0,
    },
    debug=True,
)

In [ ]:
from IPython.display import HTML, display

replay_html = env.render(mode="html")
display(HTML(replay_html))

In [ ]:
import json

with open("replay.json", "w") as f:
    json.dump(env.toJSON(), f)

## Submission

First, make sure main.py points at the policy I actually want to submit.

```bash
tar --exclude='__pycache__' --exclude='*.pyc' \
    -czf submission.tar.gz \
    main.py runner.py agent_framework policies
```
```bash
tar -tzf submission.tar.gz
```

```bash
kaggle competitions submit kaggriculture \
    -f submission.tar.gz \
    -m "
```